# A2 Pileup Autoencoder Hyperparameter Tuning

Comprehensive random search over dense and 1D CNN autoencoder architectures
for the **pileup** multi-task model (AS in the project proposal — autoencoder for
pileup events). The model must:

1. **Reconstruct** the pileup waveform from a compressed latent representation
2. **Classify** the composition of both component pulses (primary + secondary)
   as neutron (1) or photon (0) via a 2-unit sigmoid classifier head

## Key differences from a1_tune (singles tuning)

- **Data**: pileup waveforms from `pileup_waveforms.npz`, not singles
- **Labels**: 2 binary labels per event `(primary_label, secondary_label)`,
  not a single class label
- **Classifier evaluation**: accuracy = fraction where BOTH labels are correct
  (strict match). Per-label accuracy is also reported.
- **Latent dim search**: `[4, 8, 16, 32]` (broader — pileups have more complex
  structure than singles, so they may need more capacity)
- **Normalization**: pure L2 per-waveform (no min-max), same as a1_tune

All overfit safeguards from a1_tune carry over: per-trial train/val gap detection,
val-only composite score, held-out test set evaluation of top-5 finalists.

In [ ]:
import itertools
import random
import time

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, regularizers

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, confusion_matrix

import matplotlib.pyplot as plt

print(f"TensorFlow {tf.__version__}")
print(f"GPUs: {tf.config.list_physical_devices('GPU')}")

## Load and split data

Same pipeline as the baseline autoencoder notebook: load Euclidean-normalized
waveforms, split 60/20/20 train/val/test, then min-max normalize to [0, 1]
(matching the sigmoid output activation).

In [ ]:
# Load pileup waveforms and L2-normalize per-waveform.
# Labels are (primary_label, secondary_label) — both binary (0=photon, 1=neutron).
d = np.load("pileup_waveforms.npz")
X_raw = d["pileup_wf"].astype(np.float32)
y_primary   = d["primary_label"].astype(np.int32)
y_secondary = d["secondary_label"].astype(np.int32)
Y = np.column_stack([y_primary, y_secondary])  # (N, 2)

# L2-normalize each pileup waveform
X_norms = np.linalg.norm(X_raw, axis=1)
safe = np.where(X_norms == 0, 1.0, X_norms)
X = (X_raw / safe[:, None]).astype(np.float32)

# 60/20/20 train/val/test split
X_tv, X_test, Y_tv, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42,
)
X_train, X_val, Y_train, Y_val = train_test_split(
    X_tv, Y_tv, test_size=0.25, random_state=42,
)

# For the autoencoder: input = target = L2-normed waveforms
X_train_n = X_train
X_val_n   = X_val
X_test_n  = X_test

# Labels for classifier eval
y_train = Y_train
y_val   = Y_val
y_test  = Y_test

print(f"Train: {X_train_n.shape}  Val: {X_val_n.shape}  Test: {X_test_n.shape}")
print(f"Label shape: {Y_train.shape}")
print(f"L2 norm sanity: {np.linalg.norm(X_train_n, axis=1).mean():.4f}")
print(f"Value range: [{X_train_n.min():.4f}, {X_train_n.max():.4f}]")

## Search space definition

We use **random search** rather than grid search — the full grid is over
100,000 combinations and grid would be wasteful when most hyperparameters
interact only weakly. Random search is the standard approach for high
dimensional spaces (Bergstra & Bengio 2012).

In [ ]:
# ── Search space (organized by which architecture each field applies to) ──

SHARED_SPACE = {
    "latent_dim":        [4, 8, 16, 32],
    "activation":        ["relu", "elu", "leaky_relu", "swish"],
    "output_activation": ["linear"],          # L2 data has negative values, no sigmoid
    "dropout":           [0.0, 0.1, 0.2, 0.3],
    "l2_reg":            [0.0, 1e-5, 1e-4],   # dropped 1e-3 (caused collapses in a1)
    "batch_norm":        [False, True],
    "noise_std":         [0.0, 0.02, 0.05, 0.1],
    "optimizer":         ["adam", "adamw", "nadam"],
    "learning_rate":     [1e-4, 3e-4, 1e-3, 3e-3],
    "batch_size":        [256, 512, 1024],
    "loss":              ["mae", "mse", "huber"],
}

DENSE_SPACE = {
    "n_layers":       [1, 2, 3, 4, 5],
    "start_width":    [32, 64, 128, 256],
    "width_strategy": ["geometric", "linear", "funnel"],
}

CNN_SPACE = {
    "n_conv_layers":   [2, 3],
    "kernel_size":     [3, 5, 7],
    "n_filters_start": [16, 32, 64],
    "filter_strategy": ["double", "constant"],
}

MODEL_TYPES = ["dense", "cnn", "unet"]

N_TRIALS   = 100
MAX_EPOCHS = 25
PATIENCE   = 6

OVERFIT_GAP_THRESHOLD = 0.20

shared_combos = 1
for v in SHARED_SPACE.values():
    shared_combos *= len(v)
dense_combos = shared_combos
for v in DENSE_SPACE.values():
    dense_combos *= len(v)
cnn_combos = shared_combos
for v in CNN_SPACE.values():
    cnn_combos *= len(v)

print(f"Dense combos: {dense_combos:,}")
print(f"CNN combos:   {cnn_combos:,}")
print(f"Total:        {dense_combos + cnn_combos:,}")
print(f"Sampling {N_TRIALS} trials")

## Flexible model builder

Builds an autoencoder according to a hyperparameter dictionary. Supports all
the architecture/regularization options in the search space.

**Width computation:**
- `geometric` — halve the width at each layer (`start_width`, `start_width/2`, ...)
- `linear`    — evenly spaced from `start_width` down to `latent_dim`
- `funnel`    — random monotonically decreasing widths from a base set

In [ ]:
BASE_FUNNEL_WIDTHS = [256, 128, 64, 32, 16, 8]


# ── Helper functions (shared between all architectures) ─────────────────────

def compute_widths(n_layers, start_width, latent_dim, strategy, rng):
    if strategy == "geometric":
        widths = []
        w = start_width
        for _ in range(n_layers):
            if w <= latent_dim:
                w = latent_dim * 2
            widths.append(int(w))
            w = max(latent_dim, w // 2)
        return widths
    elif strategy == "linear":
        end = max(latent_dim * 2, latent_dim + 4)
        if n_layers == 1:
            return [start_width]
        return np.linspace(start_width, end, n_layers).round().astype(int).tolist()
    elif strategy == "funnel":
        candidates = [w for w in BASE_FUNNEL_WIDTHS if w >= latent_dim]
        if not candidates:
            return [latent_dim * 2] * n_layers
        widths = []
        allowed = candidates.copy()
        for _ in range(n_layers):
            w = rng.choice(allowed)
            widths.append(int(w))
            allowed = [x for x in candidates if x <= w]
            if not allowed:
                allowed = [latent_dim * 2]
        return widths
    raise ValueError(f"Unknown strategy: {strategy}")


def get_activation(name):
    if name == "leaky_relu":
        return layers.LeakyReLU(negative_slope=0.1)
    return layers.Activation(name)


def get_optimizer(name, learning_rate):
    if name == "adam":  return keras.optimizers.Adam(learning_rate=learning_rate)
    if name == "adamw": return keras.optimizers.AdamW(learning_rate=learning_rate)
    if name == "nadam": return keras.optimizers.Nadam(learning_rate=learning_rate)
    raise ValueError(f"Unknown optimizer: {name}")


def get_loss(name):
    if name == "mae":   return "mae"
    if name == "mse":   return "mse"
    if name == "huber": return keras.losses.Huber()
    raise ValueError(f"Unknown loss: {name}")


# ── Dense autoencoder builder ───────────────────────────────────────────────

def build_dense_autoencoder(hp, n_features, rng):
    widths = compute_widths(
        hp["n_layers"], hp["start_width"], hp["latent_dim"], hp["width_strategy"], rng
    )
    reg = regularizers.l2(hp["l2_reg"]) if hp["l2_reg"] > 0 else None

    enc_inputs = layers.Input(shape=(n_features,), name="encoder_input")
    x = enc_inputs
    if hp["noise_std"] > 0:
        x = layers.GaussianNoise(hp["noise_std"])(x)
    for w in widths:
        x = layers.Dense(w, kernel_regularizer=reg)(x)
        if hp["batch_norm"]:
            x = layers.BatchNormalization()(x)
        x = get_activation(hp["activation"])(x)
        if hp["dropout"] > 0:
            x = layers.Dropout(hp["dropout"])(x)
    latent = layers.Dense(hp["latent_dim"], name="latent")(x)
    latent = get_activation(hp["activation"])(latent)
    encoder = Model(enc_inputs, latent, name="encoder")

    dec_inputs = layers.Input(shape=(hp["latent_dim"],), name="decoder_input")
    x = dec_inputs
    for w in reversed(widths):
        x = layers.Dense(w, kernel_regularizer=reg)(x)
        if hp["batch_norm"]:
            x = layers.BatchNormalization()(x)
        x = get_activation(hp["activation"])(x)
        if hp["dropout"] > 0:
            x = layers.Dropout(hp["dropout"])(x)
    out = layers.Dense(n_features)(x)
    out = layers.Activation(hp["output_activation"])(out)
    decoder = Model(dec_inputs, out, name="decoder")

    ae_inputs = layers.Input(shape=(n_features,))
    autoencoder = Model(ae_inputs, decoder(encoder(ae_inputs)), name="autoencoder")
    autoencoder.compile(
        optimizer=get_optimizer(hp["optimizer"], hp["learning_rate"]),
        loss=get_loss(hp["loss"]),
    )
    return autoencoder, encoder, {"widths": widths}


# ── 1D CNN autoencoder builder (no skip connections) ────────────────────────

def build_cnn_autoencoder(hp, n_features, rng):
    n_conv = hp["n_conv_layers"]
    k = hp["kernel_size"]
    f0 = hp["n_filters_start"]
    if hp["filter_strategy"] == "double":
        filters = [f0 * (2 ** i) for i in range(n_conv)]
    else:
        filters = [f0] * n_conv

    reg = regularizers.l2(hp["l2_reg"]) if hp["l2_reg"] > 0 else None

    enc_inputs = layers.Input(shape=(n_features,), name="encoder_input")
    x = layers.Reshape((n_features, 1))(enc_inputs)
    if hp["noise_std"] > 0:
        x = layers.GaussianNoise(hp["noise_std"])(x)

    spatial = n_features
    for f in filters:
        x = layers.Conv1D(f, k, strides=2, padding="same", kernel_regularizer=reg)(x)
        if hp["batch_norm"]:
            x = layers.BatchNormalization()(x)
        x = get_activation(hp["activation"])(x)
        if hp["dropout"] > 0:
            x = layers.Dropout(hp["dropout"])(x)
        spatial = (spatial + 1) // 2

    flat_size = spatial * filters[-1]
    x = layers.Flatten()(x)
    latent = layers.Dense(hp["latent_dim"], name="latent")(x)
    latent = get_activation(hp["activation"])(latent)
    encoder = Model(enc_inputs, latent, name="encoder")

    dec_inputs = layers.Input(shape=(hp["latent_dim"],), name="decoder_input")
    x = layers.Dense(flat_size)(dec_inputs)
    x = get_activation(hp["activation"])(x)
    x = layers.Reshape((spatial, filters[-1]))(x)

    for f in reversed(filters[:-1]):
        x = layers.Conv1DTranspose(f, k, strides=2, padding="same", kernel_regularizer=reg)(x)
        if hp["batch_norm"]:
            x = layers.BatchNormalization()(x)
        x = get_activation(hp["activation"])(x)
        if hp["dropout"] > 0:
            x = layers.Dropout(hp["dropout"])(x)

    x = layers.Conv1DTranspose(1, k, strides=2, padding="same", kernel_regularizer=reg)(x)
    x = layers.Activation(hp["output_activation"])(x)
    out = layers.Lambda(lambda t: t[:, :n_features, 0], output_shape=(n_features,))(x)

    decoder = Model(dec_inputs, out, name="decoder")

    ae_inputs = layers.Input(shape=(n_features,))
    autoencoder = Model(ae_inputs, decoder(encoder(ae_inputs)), name="autoencoder")
    autoencoder.compile(
        optimizer=get_optimizer(hp["optimizer"], hp["learning_rate"]),
        loss=get_loss(hp["loss"]),
    )
    return autoencoder, encoder, {"filters": filters, "spatial": spatial}



# ── 1D CNN with U-Net skip connections ──────────────────────────────────────
#
# Same conv encoder/decoder, but with concatenation skip connections from each
# encoder layer to the corresponding decoder layer. Preserves fine timing info
# that the bottleneck would otherwise lose.  Also includes Deltoro-style
# input-to-output residual Add.

def build_unet_autoencoder(hp, n_features, rng):
    n_conv = hp["n_conv_layers"]
    k = hp["kernel_size"]
    f0 = hp["n_filters_start"]
    if hp["filter_strategy"] == "double":
        filters = [f0 * (2 ** i) for i in range(n_conv)]
    else:
        filters = [f0] * n_conv

    reg = regularizers.l2(hp["l2_reg"]) if hp["l2_reg"] > 0 else None

    # ── Encoder (save intermediate outputs for skip connections) ──
    ae_inputs = layers.Input(shape=(n_features,))
    x = layers.Reshape((n_features, 1))(ae_inputs)
    if hp["noise_std"] > 0:
        x = layers.GaussianNoise(hp["noise_std"])(x)

    skip_outputs = []
    spatial = n_features
    for f in filters:
        x = layers.Conv1D(f, k, strides=2, padding="same", kernel_regularizer=reg)(x)
        if hp["batch_norm"]:
            x = layers.BatchNormalization()(x)
        x = get_activation(hp["activation"])(x)
        if hp["dropout"] > 0:
            x = layers.Dropout(hp["dropout"])(x)
        skip_outputs.append(x)
        spatial = (spatial + 1) // 2

    # ── Bottleneck ──
    flat_size = spatial * filters[-1]
    flat = layers.Flatten()(x)
    latent = layers.Dense(hp["latent_dim"], name="latent")(flat)
    latent_act = get_activation(hp["activation"])(latent)

    x = layers.Dense(flat_size)(latent_act)
    x = get_activation(hp["activation"])(x)
    x = layers.Reshape((spatial, filters[-1]))(x)

    # ── Decoder with skip connections ──
    decoder_filters = list(reversed(filters[:-1]))
    skip_layers = list(reversed(skip_outputs[:-1]))

    for f, skip in zip(decoder_filters, skip_layers):
        x = layers.Conv1DTranspose(f, k, strides=2, padding="same", kernel_regularizer=reg)(x)
        x = layers.Concatenate()([x, skip])
        if hp["batch_norm"]:
            x = layers.BatchNormalization()(x)
        x = get_activation(hp["activation"])(x)
        if hp["dropout"] > 0:
            x = layers.Dropout(hp["dropout"])(x)

    # Final upsample back to (n_features, 1)
    x = layers.Conv1DTranspose(1, k, strides=2, padding="same", kernel_regularizer=reg)(x)
    x = layers.Activation(hp["output_activation"])(x)
    recon = layers.Lambda(lambda t: t[:, :n_features, 0], output_shape=(n_features,))(x)

    # Deltoro-style residual: decoder learns a correction to the input
    recon = layers.Add()([recon, ae_inputs])

    autoencoder = Model(ae_inputs, recon, name="autoencoder")
    encoder = Model(ae_inputs, latent_act, name="encoder")

    autoencoder.compile(
        optimizer=get_optimizer(hp["optimizer"], hp["learning_rate"]),
        loss=get_loss(hp["loss"]),
    )
    return autoencoder, encoder, {"filters": filters, "spatial": spatial, "skips": True}


# ── Dispatcher ──────────────────────────────────────────────────────────────

def build_model(hp, n_features=104, rng=None):
    rng = rng or random.Random(0)
    if hp["model_type"] == "dense":
        return build_dense_autoencoder(hp, n_features, rng)
    elif hp["model_type"] == "cnn":
        return build_cnn_autoencoder(hp, n_features, rng)
    elif hp["model_type"] == "unet":
        return build_unet_autoencoder(hp, n_features, rng)
    raise ValueError(f"Unknown model_type: {hp['model_type']}")

## Trial evaluation function

For each trial we:
1. Train the autoencoder on the train split (with early stopping on val_loss)
2. Compute reconstruction loss on the test set
3. Extract latent features for train and test
4. Train a logistic regression on the latents
5. Evaluate classification accuracy on the test set
6. Compute a composite score combining both objectives

In [ ]:
# Maximum (val_loss - train_loss) / train_loss before flagging as overfit.
OVERFIT_GAP_THRESHOLD = 0.20


def evaluate_classifier(encoder, X_train, Y_train, X_eval, Y_eval, batch_size=2048):
    # Train logistic regression on encoder features, evaluate on val.
    # For 2-label pileup classification: reports strict accuracy (both correct)
    # and per-label accuracy.
    Z_train = encoder.predict(X_train, batch_size=batch_size, verbose=0)
    Z_eval  = encoder.predict(X_eval,  batch_size=batch_size, verbose=0)

    # Train one classifier per label
    accs = []
    for label_idx in range(Y_train.shape[1]):
        clf = make_pipeline(
            StandardScaler(),
            LogisticRegression(max_iter=2000, class_weight="balanced"),
        )
        clf.fit(Z_train, Y_train[:, label_idx])
        pred = clf.predict(Z_eval)
        accs.append((pred == Y_eval[:, label_idx]).mean())

    # Strict accuracy: both labels correct
    # Re-predict to get combined result
    preds_all = np.zeros_like(Y_eval)
    for label_idx in range(Y_train.shape[1]):
        clf = make_pipeline(
            StandardScaler(),
            LogisticRegression(max_iter=2000, class_weight="balanced"),
        )
        clf.fit(Z_train, Y_train[:, label_idx])
        preds_all[:, label_idx] = clf.predict(Z_eval)

    strict_acc = (preds_all == Y_eval).all(axis=1).mean()
    primary_acc = accs[0]
    secondary_acc = accs[1]

    return strict_acc, primary_acc, secondary_acc, preds_all


def composite_score(val_loss, val_clf_acc, alpha=0.5):
    # Combined ranking metric (lower is better) — uses ONLY validation data.
    return alpha * val_loss + (1 - alpha) * (1 - val_clf_acc)


def run_trial(hp, X_train, X_val, Y_train, Y_val,
              max_epochs=MAX_EPOCHS, patience=PATIENCE, rng=None):
    # Build, train, and evaluate one config — test set is held out.
    keras.backend.clear_session()

    autoencoder, encoder, arch_info = build_model(hp, n_features=X_train.shape[1], rng=rng)
    n_params = autoencoder.count_params()

    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=patience,
            restore_best_weights=True, verbose=0,
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", patience=3, factor=0.5, verbose=0,
        ),
    ]

    t0 = time.time()
    history = autoencoder.fit(
        X_train, X_train,
        validation_data=(X_val, X_val),
        epochs=max_epochs,
        batch_size=hp["batch_size"],
        callbacks=callbacks,
        shuffle=True,
        verbose=0,
    )
    train_time = time.time() - t0

    # Train + val reconstruction loss after early stopping restored best weights
    train_loss = float(autoencoder.evaluate(X_train, X_train, batch_size=hp["batch_size"], verbose=0))
    val_loss   = float(autoencoder.evaluate(X_val,   X_val,   batch_size=hp["batch_size"], verbose=0))
    best_val   = float(np.min(history.history["val_loss"]))
    best_epoch = int(np.argmin(history.history["val_loss"]) + 1)

    # Per-trial overfit detection
    overfit_gap = (val_loss - train_loss) / max(train_loss, 1e-8)
    is_overfit = overfit_gap > OVERFIT_GAP_THRESHOLD

    # Classifier evaluated on VAL (not test)
    strict_acc, primary_acc, secondary_acc, _ = evaluate_classifier(
        encoder, X_train, Y_train, X_val, Y_val
    )

    return {
        **hp,
        **arch_info,
        "n_params": n_params,
        "best_epoch": best_epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "best_val_loss": best_val,
        "overfit_gap": overfit_gap,
        "is_overfit": is_overfit,
        "val_strict_acc": strict_acc,
        "val_primary_acc": primary_acc,
        "val_secondary_acc": secondary_acc,
        "composite_score": composite_score(best_val, strict_acc),
        "train_time_s": round(train_time, 1),
    }

## Run the random search

Each trial trains an autoencoder, then a logistic regression on its latents.
Results are saved incrementally to `a1_tune_results.csv` so you
won't lose progress if interrupted.

In [ ]:
def sample_config(rng):
    model_type = rng.choice(MODEL_TYPES)
    cfg = {"model_type": model_type}
    for k, v in SHARED_SPACE.items():
        cfg[k] = rng.choice(v)

    if model_type == "dense":
        for k, v in DENSE_SPACE.items():
            cfg[k] = rng.choice(v)
        for k in CNN_SPACE:
            cfg[k] = None
    else:
        for k, v in CNN_SPACE.items():
            cfg[k] = rng.choice(v)
        for k in DENSE_SPACE:
            cfg[k] = None
    return cfg


def config_to_key(cfg):
    return tuple(sorted(cfg.items()))


rng = random.Random(42)
results = []

sampled_configs = []
seen = set()
while len(sampled_configs) < N_TRIALS:
    cfg = sample_config(rng)
    key = config_to_key(cfg)
    if key not in seen:
        seen.add(key)
        sampled_configs.append(cfg)

n_dense = sum(1 for c in sampled_configs if c["model_type"] == "dense")
n_cnn = sum(1 for c in sampled_configs if c["model_type"] == "cnn")
print(f"Sampled {len(sampled_configs)} configs: {n_dense} dense, {n_cnn} cnn")

for trial_num, hp in enumerate(sampled_configs):
    print(f"\n{'='*78}")
    print(f"Trial {trial_num+1}/{len(sampled_configs)}  [{hp['model_type'].upper()}]")
    if hp["model_type"] == "dense":
        print(f"  arch:  latent={hp['latent_dim']} layers={hp['n_layers']} start={hp['start_width']} "
              f"strategy={hp['width_strategy']} act={hp['activation']} out={hp['output_activation']}")
    else:
        print(f"  arch:  latent={hp['latent_dim']} conv_layers={hp['n_conv_layers']} kernel={hp['kernel_size']} "
              f"f0={hp['n_filters_start']} {hp['filter_strategy']} act={hp['activation']} out={hp['output_activation']}")
    print(f"  reg:   dropout={hp['dropout']} l2={hp['l2_reg']} bn={hp['batch_norm']} noise={hp['noise_std']}")
    print(f"  opt:   {hp['optimizer']} lr={hp['learning_rate']} bs={hp['batch_size']} loss={hp['loss']}")

    try:
        result = run_trial(hp, X_train_n, X_val_n, y_train, y_val, rng=rng)
        results.append(result)
        flag = " [OVERFIT]" if result["is_overfit"] else ""
        print(f"  -> train={result['train_loss']:.5f}  val={result['val_loss']:.5f}  "
              f"gap={result['overfit_gap']:+.2%}{flag}  "
              f"strict_acc={result['val_strict_acc']:.4f}  "
              f"(pri={result['val_primary_acc']:.4f} sec={result['val_secondary_acc']:.4f})  "
              f"score={result['composite_score']:.5f}  "
              f"time={result['train_time_s']}s")
    except Exception as e:
        print(f"  -> FAILED: {e}")
        continue

    df = pd.DataFrame(results)
    df.to_csv("a2_tune_results.csv", index=False)

print(f"\n{'='*78}")
print(f"Complete: {len(results)} successful trials out of {len(sampled_configs)}")
n_overfit = sum(1 for r in results if r['is_overfit'])
print(f"Trials flagged as overfit: {n_overfit}")

## Results: top configurations

In [ ]:
df = pd.DataFrame(results)

print(f"Trials by model type:")
print(df["model_type"].value_counts().to_string())
print(f"\nOverfit trials excluded from rankings: {df['is_overfit'].sum()}")
print()

clean_df = df[~df["is_overfit"]].copy()
print(f"Eligible trials: {len(clean_df)}")
print()

# ── Top 10 overall (composite score, val-based, no overfit) ──
print("=" * 100)
print("TOP 10 OVERALL (by composite score, overfit excluded)")
print("=" * 100)
shared_cols = ["model_type", "latent_dim", "activation", "output_activation",
               "dropout", "l2_reg", "batch_norm", "noise_std",
               "optimizer", "learning_rate", "batch_size", "loss",
               "train_loss", "val_loss", "overfit_gap",
               "val_strict_acc", "val_primary_acc", "val_secondary_acc",
               "composite_score", "n_params"]
print(clean_df.sort_values("composite_score").head(10)[shared_cols].to_string())

# ── Top 5 dense ──
print("\n" + "=" * 100)
print("TOP 5 DENSE (overfit excluded)")
print("=" * 100)
dense_clean = clean_df[clean_df["model_type"] == "dense"]
if len(dense_clean) > 0:
    dense_cols = shared_cols + ["n_layers", "start_width", "width_strategy"]
    print(dense_clean.sort_values("composite_score").head(5)[dense_cols].to_string())

# ── Top 5 CNN ──
print("\n" + "=" * 100)
print("TOP 5 CNN (overfit excluded)")
print("=" * 100)
cnn_clean = clean_df[clean_df["model_type"] == "cnn"]
if len(cnn_clean) > 0:
    cnn_cols = shared_cols + ["n_conv_layers", "kernel_size", "n_filters_start", "filter_strategy"]
    print(cnn_clean.sort_values("composite_score").head(5)[cnn_cols].to_string())

# ── Architecture summary ──
print("\n" + "=" * 100)
print("ARCHITECTURE COMPARISON (mean across non-overfit trials)")
print("=" * 100)
summary = clean_df.groupby("model_type").agg(
    n_trials=("composite_score", "count"),
    mean_train_loss=("train_loss", "mean"),
    mean_val_loss=("val_loss", "mean"),
    mean_strict_acc=("val_strict_acc", "mean"),
    mean_primary_acc=("val_primary_acc", "mean"),
    mean_secondary_acc=("val_secondary_acc", "mean"),
    best_composite=("composite_score", "min"),
    mean_params=("n_params", "mean"),
)
print(summary.round(5).to_string())

## Visualization: per-hyperparameter effects

These plots show how each individual hyperparameter affects classification
accuracy. Red dot = mean for that value. Use these to spot which values
consistently work and which to drop in a follow-up search.

In [ ]:
import os
os.makedirs("figures", exist_ok=True)

# ── Per-hyperparameter effect plots, colored by model type ──
shared_hps = [
    "latent_dim", "activation", "output_activation",
    "dropout", "l2_reg", "batch_norm", "noise_std",
    "optimizer", "learning_rate", "batch_size", "loss",
]

ncols = 4
nrows = int(np.ceil(len(shared_hps) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3 * nrows))
axes = axes.flatten()

color_map = {"dense": "tab:blue", "cnn": "tab:orange"}

for ax, hp_name in zip(axes, shared_hps):
    for mt in ["dense", "cnn"]:
        sub = df[df["model_type"] == mt]
        if len(sub) == 0:
            continue
        vals = sub[hp_name].astype(str)
        ax.scatter(vals, sub["val_strict_acc"] * 100,
                   alpha=0.4, s=25, color=color_map[mt], label=mt)
    ax.set_title(hp_name, fontsize=10)
    ax.set_ylabel("Strict acc [%]", fontsize=8)
    ax.tick_params(axis="x", rotation=30, labelsize=8)
    ax.grid(True, alpha=0.3)

axes[0].legend(fontsize=8)
for j in range(len(shared_hps), len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Shared hyperparameter effects on pileup strict accuracy", fontsize=13)
plt.tight_layout()
plt.savefig("figures/a2_tune_hyperparameter_effects_shared.png", dpi=120, bbox_inches="tight")
plt.show()

# ── Dense-specific ──
dense_df = df[df["model_type"] == "dense"]
if len(dense_df) > 0:
    dense_hps = ["n_layers", "start_width", "width_strategy"]
    fig, axes = plt.subplots(1, 3, figsize=(12, 3))
    for ax, hp_name in zip(axes, dense_hps):
        vals = dense_df[hp_name].astype(str)
        for v in sorted(vals.unique()):
            mask = vals == v
            ax.scatter([v] * mask.sum(),
                       dense_df.loc[mask, "val_strict_acc"].values * 100,
                       alpha=0.5, s=25, color="tab:blue")
        means = dense_df.groupby(vals)["val_strict_acc"].mean() * 100
        ax.scatter(means.index, means.values, color="tab:red", s=80, zorder=5, label="Mean")
        ax.set_title(hp_name)
        ax.set_ylabel("Strict acc [%]")
        ax.grid(True, alpha=0.3)
    fig.suptitle("DENSE-only hyperparameter effects (pileup)", fontsize=12)
    plt.tight_layout()
    plt.savefig("figures/a2_tune_hyperparameter_effects_dense.png", dpi=120, bbox_inches="tight")
    plt.show()

# ── CNN-specific ──
cnn_df = df[df["model_type"] == "cnn"]
if len(cnn_df) > 0:
    cnn_hps = ["n_conv_layers", "kernel_size", "n_filters_start", "filter_strategy"]
    fig, axes = plt.subplots(1, 4, figsize=(16, 3))
    for ax, hp_name in zip(axes, cnn_hps):
        vals = cnn_df[hp_name].astype(str)
        for v in sorted(vals.unique()):
            mask = vals == v
            ax.scatter([v] * mask.sum(),
                       cnn_df.loc[mask, "val_strict_acc"].values * 100,
                       alpha=0.5, s=25, color="tab:orange")
        means = cnn_df.groupby(vals)["val_strict_acc"].mean() * 100
        ax.scatter(means.index, means.values, color="tab:red", s=80, zorder=5, label="Mean")
        ax.set_title(hp_name)
        ax.set_ylabel("Strict acc [%]")
        ax.grid(True, alpha=0.3)
    fig.suptitle("CNN-only hyperparameter effects (pileup)", fontsize=12)
    plt.tight_layout()
    plt.savefig("figures/a2_tune_hyperparameter_effects_cnn.png", dpi=120, bbox_inches="tight")
    plt.show()

## Pareto front: reconstruction vs classification

Models in the lower-right region (low reconstruction loss AND high
classification accuracy) are the best overall. Use this plot to spot the
tradeoff: pure reconstruction quality doesn't always give the best features
for classification.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: train_loss vs val_loss (overfit detection)
ax = axes[0]
for mt, color in [("dense", "tab:blue"), ("cnn", "tab:orange")]:
    sub = df[df["model_type"] == mt]
    if len(sub) == 0:
        continue
    ok = sub[~sub["is_overfit"]]
    bad = sub[sub["is_overfit"]]
    ax.scatter(ok["train_loss"], ok["val_loss"], color=color, alpha=0.7,
               s=60, edgecolors="k", label=f"{mt} ok")
    if len(bad) > 0:
        ax.scatter(bad["train_loss"], bad["val_loss"], color=color, alpha=0.7,
                   s=80, edgecolors="red", linewidth=2, marker="X",
                   label=f"{mt} overfit")
lim = max(df.train_loss.max(), df.val_loss.max()) * 1.05
ax.plot([0, lim], [0, lim], "k--", alpha=0.5, label="y = x")
ax.set_xlabel("Train reconstruction loss")
ax.set_ylabel("Val reconstruction loss")
ax.set_title("Train vs Val loss (overfit check)")
ax.set_xlim(0, lim)
ax.set_ylim(0, lim)
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Right: val recon vs val strict accuracy (ranking surface)
ax = axes[1]
clean_df = df[~df["is_overfit"]]
for mt, color in [("dense", "tab:blue"), ("cnn", "tab:orange")]:
    sub = clean_df[clean_df["model_type"] == mt]
    ax.scatter(sub["val_loss"], sub["val_strict_acc"] * 100,
               s=60, alpha=0.7, edgecolors="k", color=color, label=mt)
ax.set_xlabel("Val reconstruction loss")
ax.set_ylabel("Val strict accuracy [%]")
ax.set_title("Val reconstruction vs classification (non-overfit)")
ax.grid(True, alpha=0.3)

for mt, color in [("dense", "blue"), ("cnn", "darkorange")]:
    sub = clean_df[clean_df["model_type"] == mt]
    if len(sub) == 0:
        continue
    best = sub.loc[sub["composite_score"].idxmin()]
    ax.scatter([best["val_loss"]], [best["val_strict_acc"] * 100],
               marker="*", s=400, color=color, edgecolors="k",
               label=f"Best {mt}")

ax.legend()
plt.tight_layout()
plt.savefig("figures/a2_tune_pareto.png", dpi=120, bbox_inches="tight")
plt.show()

## Best configuration summary

In [ ]:
def print_config(label, sub_df):
    if len(sub_df) == 0:
        print(f"\n{label}: no trials")
        return
    best = sub_df.loc[sub_df["composite_score"].idxmin()]
    print("=" * 70)
    print(f"BEST {label}")
    print("=" * 70)
    print(f"  model_type:        {best['model_type']}")
    print(f"  latent_dim:        {int(best['latent_dim'])}")
    if best["model_type"] == "dense":
        print(f"  n_layers:          {int(best['n_layers'])}")
        print(f"  start_width:       {int(best['start_width'])}")
        print(f"  width_strategy:    {best['width_strategy']}")
    else:
        print(f"  n_conv_layers:     {int(best['n_conv_layers'])}")
        print(f"  kernel_size:       {int(best['kernel_size'])}")
        print(f"  n_filters_start:   {int(best['n_filters_start'])}")
        print(f"  filter_strategy:   {best['filter_strategy']}")
    print(f"  activation:        {best['activation']}")
    print(f"  output_activation: {best['output_activation']}")
    print(f"  dropout:           {best['dropout']}")
    print(f"  l2_reg:            {best['l2_reg']}")
    print(f"  batch_norm:        {best['batch_norm']}")
    print(f"  noise_std:         {best['noise_std']}")
    print(f"  optimizer:         {best['optimizer']}")
    print(f"  learning_rate:     {best['learning_rate']}")
    print(f"  batch_size:        {int(best['batch_size'])}")
    print(f"  loss:              {best['loss']}")
    print()
    print(f"  n_params:          {int(best['n_params']):,}")
    print(f"  train_loss:        {best['train_loss']:.5f}")
    print(f"  val_loss:          {best['val_loss']:.5f}")
    print(f"  overfit_gap:       {best['overfit_gap']:+.2%}")
    print(f"  val_strict_acc:    {best['val_strict_acc']:.4f}")
    print(f"  val_primary_acc:   {best['val_primary_acc']:.4f}")
    print(f"  val_secondary_acc: {best['val_secondary_acc']:.4f}")
    print(f"  composite_score:   {best['composite_score']:.5f}")
    print(f"  best_epoch:        {int(best['best_epoch'])}")


def hp_dict_from_row(row):
    keys = list(SHARED_SPACE) + list(DENSE_SPACE) + list(CNN_SPACE) + ["model_type"]
    return {k: (None if pd.isna(row[k]) else row[k]) for k in keys if k in row.index}


clean_df = df[~df["is_overfit"]].copy()

print_config("OVERALL (val-only)", clean_df)
print()
print_config("DENSE (val-only)", clean_df[clean_df["model_type"] == "dense"])
print()
print_config("CNN (val-only)", clean_df[clean_df["model_type"] == "cnn"])

# ── Honest test-set evaluation of top-5 finalists ──
print("\n" + "=" * 70)
print("FINAL HONEST EVALUATION ON HELD-OUT TEST SET")
print("=" * 70)
print("Re-training top-5 candidates from scratch, reporting on test set.")
print()

top5 = clean_df.sort_values("composite_score").head(5)
final_results = []

for rank, (_, row) in enumerate(top5.iterrows(), 1):
    hp = hp_dict_from_row(row)
    print(f"\nFinalist {rank}: {hp['model_type']} latent={int(hp['latent_dim'])} "
          f"composite={row['composite_score']:.5f}")
    keras.backend.clear_session()
    rng2 = random.Random(rank * 1000)
    autoencoder, encoder, _ = build_model(hp, n_features=X_train_n.shape[1], rng=rng2)
    cb = [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE,
                                       restore_best_weights=True, verbose=0),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", patience=3, factor=0.5, verbose=0),
    ]
    autoencoder.fit(X_train_n, X_train_n, validation_data=(X_val_n, X_val_n),
                    epochs=MAX_EPOCHS, batch_size=int(hp["batch_size"]),
                    callbacks=cb, shuffle=True, verbose=0)

    test_recon_loss = float(autoencoder.evaluate(X_test_n, X_test_n,
                                                  batch_size=int(hp["batch_size"]), verbose=0))
    test_strict, test_pri, test_sec, _ = evaluate_classifier(
        encoder, X_train_n, y_train, X_test_n, y_test
    )
    print(f"  test_recon_loss   = {test_recon_loss:.5f}")
    print(f"  test_strict_acc   = {test_strict:.4f}  (val was {row['val_strict_acc']:.4f})")
    print(f"  test_primary_acc  = {test_pri:.4f}")
    print(f"  test_secondary_acc= {test_sec:.4f}")
    final_results.append({
        "rank": rank,
        "model_type": hp["model_type"],
        "latent_dim": int(hp["latent_dim"]),
        "val_strict_acc": row["val_strict_acc"],
        "test_strict_acc": test_strict,
        "test_primary_acc": test_pri,
        "test_secondary_acc": test_sec,
        "test_recon_loss": test_recon_loss,
    })

print("\n" + "=" * 70)
print("FINALIST SUMMARY")
print("=" * 70)
final_df = pd.DataFrame(final_results)
print(final_df.to_string(index=False))

df.to_csv("a2_tune_results.csv", index=False)
final_df.to_csv("a2_tune_finalists.csv", index=False)
print("\nSaved a2_tune_results.csv and a2_tune_finalists.csv")